In [1]:
import os
import json
import time
import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

# 1. THE SINGLE ROW BLUEPRINT
class DesignPalette(BaseModel):
    mood_industry: str = Field(description="The exact mood or industry")
    primary_color: str = Field(description="Hex code for primary brand color")
    secondary_color: str = Field(description="Hex code for secondary color")
    background_color: str = Field(description="Hex code for app background")
    text_color: str = Field(description="Hex code for main readable text")
    accent_color: str = Field(description="Hex code for buttons or highlights")
    heading_font: str = Field(description="A Google Font name for headings")
    body_font: str = Field(description="A Google Font name for body text")

# 2. THE NEW BATCH BLUEPRINT (This is the magic part)
class PaletteBatch(BaseModel):
    palettes: list[DesignPalette]

client = genai.Client(api_key="AQ.Ab8RN6JKPw5cYB1StICP0gSsYrZkmMSVKlLayVw2iDbfp6uZUQ")

# 3. LOAD MASSIVE LIST FROM TEXT FILE
print("Loading moods from text file...")
with open("moods.txt", "r", encoding="utf-8") as file:
    # This reads every line, removes extra spaces, and ignores blank lines
    moods_list = [line.strip() for line in file.readlines() if line.strip()]

print(f"Loaded {len(moods_list)} moods to generate!")

# 4. CHUNK THE LIST INTO GROUPS OF 10
# This groups our long list into smaller chunks so we don't overwhelm the AI
chunk_size = 10
mood_chunks = [moods_list[i:i + chunk_size] for i in range(0, len(moods_list), chunk_size)]

print("Starting batched generation...")

for chunk in mood_chunks:
    # We join the 10 moods into a single string separated by commas
    moods_string = ", ".join(chunk)
    print(f"Generating batch for: {moods_string}")
    
    prompt = f"Act as an expert UI/UX designer. Create exactly {len(chunk)} distinct design systems, one for each of these industries: {moods_string}"
    
    try:
        response = client.models.generate_content(
            model='gemini-3.5-flash', # Updated model version here
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=PaletteBatch,
                temperature=0.7
            )
        )
        
        # Parse the JSON and extract the list of palettes
        batch_data = json.loads(response.text)
        
        # Add these 10 new palettes to our master list
        all_palettes.extend(batch_data["palettes"])
        
        # We still sleep, but now every sleep gets us 10 rows instead of 1!
        time.sleep(4) 
        
    except Exception as e:
        print(f"Error on chunk: {e}")
        
        # Parse the JSON and extract the list of palettes
        batch_data = json.loads(response.text)
        
        # Add these 10 new palettes to our master list
        all_palettes.extend(batch_data["palettes"])
        
        # We still sleep, but now every sleep gets us 10 rows instead of 1!
        time.sleep(4) 
        
    except Exception as e:
        print(f"Error on chunk: {e}")

# 5. SAVE TO CSV
df = pd.DataFrame(all_palettes)
df.to_csv("ui_dataset_batched.csv", index=False)
print(f"Finished! Saved {len(df)} rows to ui_dataset_batched.csv")

Loading moods from text file...
Loaded 15 moods to generate!
Starting batched generation...
Generating batch for: Eco-friendly organic grocery store, Neon cyberpunk gaming forum, Vintage 1950s diner POS system, Corporate financial dashboard, Playful kindergarten learning app, Dark mode developer portfolio, Luxury minimalist real estate agency, High-energy modern fitness tracker, Calm and trustworthy online therapy portal, Bold and aggressive sports news site
Error on chunk: name 'all_palettes' is not defined


NameError: name 'all_palettes' is not defined